# 05 — Adaptive Sampling & Pareto Analysis

Simulate stopping policies on pre-generated candidates, run ablations,
train CSC LoRA, build Pareto curves.

In [ ]:
import sys, os
sys.path.insert(0, '/kaggle/working/SLM-PRJ')
os.chdir('/kaggle/working/SLM-PRJ')

from pathlib import Path
import json

## 1. Offline Policy Simulation

Replay stopping policies on cached candidates (N=32 per question, 100+ permutations).

In [ ]:
from benchmarks.bird_loader import load_bird_dev, get_db_path
from research.exec_runner import load_exec_results

DATA_DIR = Path('/kaggle/input/bird-benchmark')
EXEC_FILE = Path('results/candidates/exec_dev.jsonl')
RESULTS_DIR = Path('results/policies')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

records = load_exec_results(EXEC_FILE)
db_path_fn = lambda db_id: get_db_path(DATA_DIR, db_id)
print(f'Loaded {len(records)} questions with candidates')

In [ ]:
from research.policies.fixed_k import FixedKPolicy
from research.policies.beta_stopping import BetaStoppingPolicy
from research.policies.margin_entropy import MarginEntropyPolicy
from research.simulator import simulate_multiple_policies

policies = {
    'fixed_k4': FixedKPolicy(k=4),
    'fixed_k8': FixedKPolicy(k=8),
    'fixed_k16': FixedKPolicy(k=16),
    'fixed_k32': FixedKPolicy(k=32),
    'beta_095': BetaStoppingPolicy(threshold=0.95, min_k=4, max_k=32),
    'beta_090': BetaStoppingPolicy(threshold=0.90, min_k=4, max_k=32),
    'margin_3': MarginEntropyPolicy(margin=3, entropy_threshold=1.0, min_k=4, max_k=32),
    'margin_5': MarginEntropyPolicy(margin=5, entropy_threshold=0.8, min_k=4, max_k=32),
}

sim_results = simulate_multiple_policies(records, db_path_fn, policies, n_permutations=100)

print(f'{"Policy":<15} {"EX":>8} {"CI95":>12} {"Avg_K":>8}')
print('-' * 45)
for name, res in sim_results.items():
    ci = f'[{res["ci95_lower"]:.4f},{res["ci95_upper"]:.4f}]'
    print(f'{name:<15} {res["ex_mean"]:>8.4f} {ci:>12} {res["avg_samples"]:>8.1f}')

In [ ]:
with open(RESULTS_DIR / 'simulation_results.json', 'w') as f:
    json.dump(sim_results, f, indent=2)
print('Saved simulation results')

## 2. Learned Stopper (Logistic + XGBoost)

In [ ]:
from research.policies.learned_stopper import LearnedStopperPolicy
from research.calibration import compute_calibration_data

calib_data = compute_calibration_data(records, db_path_fn)

for model_type in ['logistic', 'xgboost']:
    policy = LearnedStopperPolicy(model_type=model_type, min_k=4, max_k=32)
    policy.fit(calib_data)
    
    result = simulate_multiple_policies(
        records, db_path_fn,
        {f'learned_{model_type}': policy},
        n_permutations=100,
    )
    res = result[f'learned_{model_type}']
    print(f'{model_type:10s}  EX={res["ex_mean"]:.4f}  Avg_K={res["avg_samples"]:.1f}')
    sim_results[f'learned_{model_type}'] = res

with open(RESULTS_DIR / 'simulation_results.json', 'w') as f:
    json.dump(sim_results, f, indent=2)

## 3. Candidate Quality Ablation

In [ ]:
from research.candidate_quality import run_ablation

ablation_results = run_ablation(records, db_path_fn)

print(f'{"Config":<30} {"EX":>8} {"N":>6}')
print('-' * 46)
for label, metrics in ablation_results.items():
    print(f'{label:<30} {metrics["ex"]:>8.4f} {metrics["n"]:>6}')

with open(RESULTS_DIR / 'ablation_results.json', 'w') as f:
    json.dump(ablation_results, f, indent=2)

## 4. CSC LoRA Training (GPU required)

Train a merge-correct LoRA adapter from top-2 candidate groups.

In [ ]:
from research.csc_lora import build_csc_training_data

CSC_DIR = Path('results/csc')
CSC_DIR.mkdir(parents=True, exist_ok=True)

stats = build_csc_training_data(records, db_path_fn, CSC_DIR / 'csc_train.jsonl')
print(f'CSC training pairs: {stats["total_pairs"]}')
print(f'Gold in top-2: {stats["gold_in_top2"]} ({stats["gold_in_top2"]/max(stats["total_pairs"],1)*100:.1f}%)')

In [ ]:
from research.csc_lora import train_csc_lora

BEST_ADAPTER = 'results/lora/best_adapter/merged'  # adjust path

csc_result = train_csc_lora(
    training_data_path=CSC_DIR / 'csc_train.jsonl',
    base_adapter_path=BEST_ADAPTER,
    output_path=CSC_DIR / 'csc_adapter',
    lora_rank=16,
    epochs=3,
    lr=2e-4,
)
print(f'CSC adapter trained: {csc_result}')

## 5. Three-Action Router Simulation

In [ ]:
from research.policies.three_action_router import ThreeActionRouterPolicy

router_configs = {
    'router_default': ThreeActionRouterPolicy(
        stop_margin=3, csc_threshold=0.6, min_k=4, max_k=32,
    ),
    'router_aggressive': ThreeActionRouterPolicy(
        stop_margin=2, csc_threshold=0.5, min_k=4, max_k=24,
    ),
    'router_conservative': ThreeActionRouterPolicy(
        stop_margin=5, csc_threshold=0.7, min_k=8, max_k=32,
    ),
}

router_results = simulate_multiple_policies(
    records, db_path_fn, router_configs, n_permutations=100,
)

print(f'{"Router":<25} {"EX":>8} {"Avg_K":>8} {"CSC_calls":>10}')
print('-' * 53)
for name, res in router_results.items():
    print(f'{name:<25} {res["ex_mean"]:>8.4f} {res["avg_samples"]:>8.1f} {res.get("csc_calls", "N/A"):>10}')
    sim_results[name] = res

with open(RESULTS_DIR / 'simulation_results.json', 'w') as f:
    json.dump(sim_results, f, indent=2)

## 6. Pareto Analysis

Aggregate all experiments into EX vs cost Pareto curves.

In [ ]:
from research.pareto import build_pareto_table, pareto_frontier, save_pareto_report, plot_pareto

all_experiments = []
for name, res in sim_results.items():
    all_experiments.append({
        'name': name,
        'ex_mean': res.get('ex_mean', 0),
        'avg_samples': res.get('avg_samples', 0),
        'avg_tokens': res.get('avg_tokens', 0),
        'avg_seconds': res.get('avg_seconds', 0),
    })

rows = build_pareto_table(all_experiments)

save_pareto_report(rows, 'results/final/pareto_report.json')
print('Pareto report saved')

frontier = pareto_frontier(rows)
print(f'\nPareto frontier ({len(frontier)} points):')
for r in frontier:
    print(f'  {r["name"]}: EX={r["ex"]:.4f}, samples={r["avg_samples"]:.1f}')

In [ ]:
plot_pareto(rows, x_key='avg_samples', y_key='ex', output_path='results/final/pareto_samples.png')
plot_pareto(rows, x_key='avg_tokens', y_key='ex', output_path='results/final/pareto_tokens.png')
print('Pareto plots saved to results/final/')

## 7. Summary Table

In [ ]:
print(f'{"Experiment":<25} {"EX":>8} {"Avg_K":>8} {"CI95_lo":>8} {"CI95_hi":>8}')
print('=' * 59)

sorted_results = sorted(sim_results.items(), key=lambda x: x[1].get('ex_mean', 0), reverse=True)
for name, res in sorted_results:
    print(f'{name:<25} {res.get("ex_mean",0):>8.4f} {res.get("avg_samples",0):>8.1f} '
          f'{res.get("ci95_lower",0):>8.4f} {res.get("ci95_upper",0):>8.4f}')

best = sorted_results[0]
print(f'\nBest policy: {best[0]} with EX={best[1]["ex_mean"]:.4f} using {best[1]["avg_samples"]:.1f} avg samples')